In [ ]:
from pyspark.sql import SparkSession, functions as F
from IPython.display import display

spark = (SparkSession.builder
    .appName("olist-04-integration")
    .config("spark.sql.warehouse.dir", "hdfs:///user/hive/warehouse")
    .enableHiveSupport()
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

RAW    = "hdfs:///data/olist/raw"
BUCKET = "gs://olist-sujinan-2026"
DB     = "olist"

def show(df, n=20):
    display(df.limit(n).toPandas())

def pdf(df, n=1000):
    return df.limit(n).toPandas()

print("warehouse:", spark.conf.get("spark.sql.warehouse.dir"))

In [ ]:
orders    = spark.table(f"{DB}.silver_orders")
items     = spark.table(f"{DB}.silver_order_items")
products  = spark.table(f"{DB}.silver_products")
sellers   = spark.table(f"{DB}.silver_sellers")
customers = spark.table(f"{DB}.silver_customers")
geo       = spark.table(f"{DB}.silver_geolocation")
reviews   = spark.table(f"{DB}.silver_reviews")
payments  = spark.table(f"{DB}.silver_payments")

for name, df in [("orders", orders), ("items", items), ("products", products),
                 ("sellers", sellers), ("customers", customers), ("geo", geo),
                 ("reviews", reviews), ("payments", payments)]:
    print(f"{name:<11s} {df.count():>9,} rows")

# Reduce the number of lines to one per order before joining

In [ ]:
payments_agg = (payments
    .groupBy("order_id")
    .agg(
        F.round(F.sum("payment_value"), 2).alias("payment_total"),
        F.count("*").alias("payment_count"),
        F.max("payment_installments").alias("max_installments"),
        F.first("payment_method").alias("payment_method"),
    ))

reviews_agg = (reviews
    .groupBy("order_id")
    .agg(
        F.round(F.avg("review_score"), 2).alias("review_score"),
        F.count("*").alias("review_count"),
    ))

print(f"payments  {payments.count():,} > {payments_agg.count():,}")
print(f"reviews   {reviews.count():,} > {reviews_agg.count():,}")

# Join all 8 tables into a single table

In [ ]:
full_orders = (
    orders
      .join(items,                    "order_id",    "inner")
      .join(products,                 "product_id",  "inner")
      .join(F.broadcast(sellers),     "seller_id",   "inner")
      .join(customers,                "customer_id", "inner")
      .join(F.broadcast(geo),         "customer_zip_code_prefix", "left")
      .join(F.broadcast(reviews_agg), "order_id",    "left")
      .join(F.broadcast(payments_agg),"order_id",    "left")
)

full_orders.cache()
print(f"items {items.count():,} > full_orders {full_orders.count():,}")

# Business Aggregation

## Customer Value - Total Spent and AOV

In [ ]:
customer_spending = (full_orders
    .groupBy("customer_unique_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum("price"), 2).alias("total_spent"),
        F.round(F.avg("price"), 2).alias("avg_item_price"),
    )
    .withColumn("aov", F.round(F.col("total_spent") / F.col("total_orders"), 2))
    .orderBy(F.desc("total_spent"))
)

show(customer_spending)

## Seller Performance

In [ ]:
seller_performance = (full_orders
    .groupBy("seller_id")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.round(F.sum("price"), 2).alias("total_revenue"),
        F.round(F.avg("review_score"), 2).alias("avg_review_score"),
        F.round(F.stddev("price"), 2).alias("price_variability"),
    )
    .orderBy(F.desc("total_revenue"))
)
show(seller_performance)

## Product popularity

In [ ]:
product_metrics = (full_orders
    .groupBy("product_id", "product_category_name_english")
    .agg(
        F.count("order_id").alias("total_sales"),
        F.round(F.sum("price"), 2).alias("total_revenue"),
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.round(F.stddev("price"), 2).alias("price_volatility"),
        F.size(F.collect_set("seller_id")).alias("unique_sellers"),
    )
    .orderBy(F.desc("total_sales"))
)
show(product_metrics)

## Monthly outlook

In [ ]:
monthly_trend = (full_orders
    .withColumn("order_month", F.date_trunc("month", "order_purchase_timestamp"))
    .groupBy("order_month")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.round(F.sum("price"), 2).alias("total_revenue"),
        F.round(F.avg("price"), 2).alias("aov"),
        F.min("price").alias("min_order_value"),
        F.max("price").alias("max_order_value"),
    )
    .orderBy("order_month")
)

show(monthly_trend, 30)

trend = pdf(monthly_trend, 30)
trend.plot.line(x="order_month", y="total_revenue", figsize=(12, 4))

## Customer repeat purchases

In [ ]:
customer_retention = (full_orders
    .groupBy("customer_unique_id")
    .agg(
        F.min("order_purchase_timestamp").alias("first_order_date"),
        F.max("order_purchase_timestamp").alias("last_order_date"),
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.avg("price"), 2).alias("aov"),
    )
    .withColumn("customer_lifespan_days",
                F.datediff("last_order_date", "first_order_date"))
    .orderBy(F.desc("total_orders"))
)

show(customer_retention)

In [ ]:
show(customers.agg(
    F.countDistinct("customer_id").alias("customer_id"),
    F.countDistinct("customer_unique_id").alias("customer_unique_id"),
))

# Window function to rank products for each seller

In [ ]:
from pyspark.sql.window import Window

w_seller = Window.partitionBy("seller_id").orderBy(F.desc("price"))

top_products_per_seller = (full_orders
    .withColumn("rank_in_seller", F.rank().over(w_seller))
    .filter(F.col("rank_in_seller") <= 5)
    .select("seller_id", "product_id", "price", "rank_in_seller")
    .orderBy("seller_id", "rank_in_seller")
)

show(top_products_per_seller)

# Enrichment - Add frequently used columns at the destination

In [ ]:
full_orders = (full_orders
    .withColumn("is_delivered",
                F.when(F.col("order_status") == "delivered", 1).otherwise(0))
    .withColumn("is_canceled",
                F.when(F.col("order_status") == "canceled", 1).otherwise(0))

    # Actual revenue per transaction = Product price + Shipping cost
    .withColumn("order_revenue", F.col("price") + F.col("freight_value"))

    # Ordered hours - used for call center workforce planning.
    .withColumn("order_hour", F.hour("order_purchase_timestamp"))

    # Weekdays vs Weekends (dayofweek 1 = Sunday, 7 = Saturday)
    .withColumn("order_day_type",
                F.when(F.dayofweek("order_purchase_timestamp").isin(1, 7),
                       "weekend").otherwise("weekday"))
)

## Group the customers and merge back into the main table

In [ ]:
segments = customer_spending.select(
    "customer_unique_id",
    F.when(F.col("aov") >= 1200, "high_value")
     .when(F.col("aov") >= 500,  "medium_value")
     .otherwise("low_value")
     .alias("customer_segment")
)

full_orders = full_orders.drop("customer_segment")
full_orders = full_orders.join(F.broadcast(segments), "customer_unique_id", "left")

show(full_orders.groupBy("customer_segment")
     .agg(F.countDistinct("order_id").alias("orders"),
          F.count("*").alias("items"),
          F.round(F.sum("order_revenue"), 2).alias("revenue")))

In [ ]:
print("items                ", f"{items.count():>9,}")
print("full_orders          ", f"{full_orders.count():>9,}")
print("payments rows        ", f"{payments.count():>9,}",
      " distinct orders:", f"{payments.select('order_id').distinct().count():,}")
print("reviews rows         ", f"{reviews.count():>9,}",
      " distinct orders:", f"{reviews.select('order_id').distinct().count():,}")

In [ ]:
show(orders.join(payments.select("order_id").distinct(), "order_id", "left_anti"))

In [ ]:
gold = {
    "full_orders":             (full_orders, 4),
    "customer_spending":       (customer_spending, 1),
    "seller_performance":      (seller_performance, 1),
    "product_metrics":         (product_metrics, 1),
    "monthly_trend":           (monthly_trend, 1),
    "customer_retention":      (customer_retention, 1),
    "top_products_per_seller": (top_products_per_seller, 1),
}

for name, (df, parts) in gold.items():
    out = df.coalesce(parts)
    out.write.mode("overwrite").format("parquet").saveAsTable(f"{DB}.gold_{name}")
    out.write.mode("overwrite").parquet(f"{BUCKET}/gold/{name}")
    print(f"saved gold_{name}")

print()
for name in gold:
    n = spark.table(f"{DB}.gold_{name}").count()
    print(f"gold_{name:<24s} {n:>9,} rows")

full_orders.unpersist()